# Übung 04 — Datenformate CSV, JSON, XML (Lösung)

Bezug: *Factsheet Data Science*, Kapitel 11 (Datenformate) und 12.5 (Stolpersteine).

Im Ordner `Data/` liegen dieselben Mitgliederdaten in drei Formaten:
`mitglieder.csv`, `mitglieder.json`, `mitglieder.xml`. Öffne sie zuerst im Editor und vergleiche den Aufbau.

## Bibliotheken

In [ ]:
import os
import json
import pandas as pd

os.makedirs("output", exist_ok=True)

## 4.1 Drei Formate — eine Tabelle
Lies alle drei Dateien mit pandas ein (`read_csv`, `read_json`, `read_xml`) in `df_csv`, `df_json`, `df_xml`.
Prüfe anschliessend, dass alle drei denselben Inhalt haben.

In [ ]:
df_csv = pd.read_csv("Data/mitglieder.csv", encoding="utf-8")
df_json = pd.read_json("Data/mitglieder.json")
df_xml = pd.read_xml("Data/mitglieder.xml")   # benötigt die Bibliothek lxml

df_csv

In [ ]:
# Selbstkontrolle 4.1
assert df_csv.shape == df_json.shape == df_xml.shape == (6, 4)
assert df_csv.values.tolist() == df_json.values.tolist() == df_xml.values.tolist()
print("4.1 richtig ✔")

## 4.2 Verschachteltes JSON
CSV kann nur flache Tabellen abbilden — JSON auch Verschachtelungen. Der Text `roh` stammt von einer API.
1. Parse ihn mit `json.loads` in die Variable `daten` (eine Python-Liste von Dictionaries).
2. Lies Mias E-Mail-Adresse aus → `mia_email`.
3. Mache mit `pd.json_normalize` eine flache Tabelle daraus → `flach`. Welche Spaltennamen entstehen?

In [ ]:
roh = '''
[
  {"name": "Anna", "team": "U15", "lizenzen": ["Swiss Badminton", "J+S"]},
  {"name": "Mia", "team": "Damen 1",
   "kontakt": {"email": "mia@example.ch", "telefon": "079 000 00 00"}}
]
'''

In [ ]:
daten = json.loads(roh)
mia_email = daten[1]["kontakt"]["email"]
flach = pd.json_normalize(daten)
flach
# Aus dem Unterobjekt "kontakt" werden die Spalten "kontakt.email" und "kontakt.telefon".
# Bei Anna fehlen sie -> NaN. Die Liste "lizenzen" bleibt als Liste in einer Zelle stehen.

In [ ]:
# Selbstkontrolle 4.2
assert isinstance(daten, list) and mia_email == "mia@example.ch"
assert "kontakt.email" in flach.columns
print("4.2 richtig ✔")

## 4.3 JSON schreiben
Speichere `df_csv` als `output/mitglieder_export.json`: eine Liste von Objekten (eines pro Zeile),
Umlaute erhalten, lesbar eingerückt. Tipp: `orient`, `force_ascii`, `indent`.

In [ ]:
df_csv.to_json("output/mitglieder_export.json", orient="records", force_ascii=False, indent=2)

In [ ]:
# Selbstkontrolle 4.3
with open("output/mitglieder_export.json", encoding="utf-8") as f:
    export = json.load(f)
assert isinstance(export, list) and export[0]["name"] == "Anna" and len(export) == 6
print("4.3 richtig ✔")

## 4.4 CSV für Excel (Schweiz) und das Trennzeichen-Problem
1. Speichere `df_csv` als `output/mitglieder_excel.csv` mit Semikolon und `utf-8-sig`, ohne Index.
2. Lies die Datei **ohne** Angabe von `sep` wieder ein → `falsch`. Wie viele Spalten hat sie?
3. Lies sie korrekt ein → `richtig`.

In [ ]:
df_csv.to_csv("output/mitglieder_excel.csv", sep=";", index=False, encoding="utf-8-sig")

falsch = pd.read_csv("output/mitglieder_excel.csv", encoding="utf-8-sig")
richtig = pd.read_csv("output/mitglieder_excel.csv", sep=";", encoding="utf-8-sig")
print(falsch.shape, richtig.shape)
# falsch hat nur 1 Spalte: pandas erwartet Kommas und findet keine.

In [ ]:
# Selbstkontrolle 4.4
assert falsch.shape[1] == 1 and richtig.shape == (6, 4)
print("4.4 richtig ✔")

## 4.5 Encoding: Woher kommt "ZÃ¼rich"?
Wandle den Text `"Zürich"` mit `.encode("utf-8")` in Bytes um und lies diese Bytes **falsch** mit
`.decode("latin-1")` wieder ein → `kaputt`. Repariere den Text anschliessend → `repariert`.

In [ ]:
kaputt = "Zürich".encode("utf-8").decode("latin-1")
repariert = kaputt.encode("latin-1").decode("utf-8")
print(kaputt, "->", repariert)
# Darum beim Einlesen immer das richtige Encoding angeben: encoding="utf-8" bzw. "utf-8-sig".

In [ ]:
# Selbstkontrolle 4.5
assert kaputt == "ZÃ¼rich" and repariert == "Zürich"
print("4.5 richtig ✔")

## 4.6 XML schreiben
Speichere nur die Mitglieder des Teams `U15` als `output/u15.xml`. Das Wurzelelement soll `mitglieder`,
jedes Mitglied ein Element `mitglied` heissen, ohne Index. Tipp: `to_xml(..., root_name=, row_name=, index=)`.

In [ ]:
u15 = df_csv[df_csv["team"] == "U15"]
u15.to_xml("output/u15.xml", root_name="mitglieder", row_name="mitglied", index=False)

In [ ]:
# Selbstkontrolle 4.6
kontrolle = pd.read_xml("output/u15.xml")
assert list(kontrolle["name"]) == ["Anna", "Luca", "Jonas"]
with open("output/u15.xml", encoding="utf-8") as f:
    assert "<mitglied>" in f.read()
print("4.6 richtig ✔")

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')